In [2]:
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from sklearn.pipeline import Pipeline
pd.options.plotting.backend = 'plotly'


from dsc80_utils import * # Feel free to uncomment and use this.

In [3]:
import plotly.io as pio
pio.renderers.default = 'jupyterlab'

In [4]:
from datetime import datetime, time

In [5]:
pd.set_option('display.max_columns', None)

In [6]:
original = pd.read_csv('data/power_outage.csv')

In [7]:
original.head()

,OBS,YEAR,MONTH,U.S._STATE,POSTAL.CODE,NERC.REGION,CLIMATE.REGION,ANOMALY.LEVEL,CLIMATE.CATEGORY,OUTAGE.START.DATE,OUTAGE.START.TIME,OUTAGE.RESTORATION.DATE,OUTAGE.RESTORATION.TIME,CAUSE.CATEGORY,CAUSE.CATEGORY.DETAIL,HURRICANE.NAMES,OUTAGE.DURATION,DEMAND.LOSS.MW,CUSTOMERS.AFFECTED,RES.PRICE,COM.PRICE,IND.PRICE,TOTAL.PRICE,RES.SALES,COM.SALES,IND.SALES,TOTAL.SALES,RES.PERCEN,COM.PERCEN,IND.PERCEN,RES.CUSTOMERS,COM.CUSTOMERS,IND.CUSTOMERS,TOTAL.CUSTOMERS,RES.CUST.PCT,COM.CUST.PCT,IND.CUST.PCT,PC.REALGSP.STATE,PC.REALGSP.USA,PC.REALGSP.REL,PC.REALGSP.CHANGE,UTIL.REALGSP,TOTAL.REALGSP,UTIL.CONTRI,PI.UTIL.OFUSA,POPULATION,POPPCT_URBAN,POPPCT_UC,POPDEN_URBAN,POPDEN_UC,POPDEN_RURAL,AREAPCT_URBAN,AREAPCT_UC,PCT_LAND,PCT_WATER_TOT,PCT_WATER_INLAND
0,1,2011,7.0,Minnesota,MN,MRO,East North Central,-0.3,normal,"Friday, July 01, 2011",5:00:00 PM,"Sunday, July 03, 2011",8:00:00 PM,severe weather,NaN,NaN,3060.0,NaN,70000.0,11.60,9.18,6.81,9.28,2.33e+06,2.11e+06,2.11e+06,6.56e+06,35.55,32.23,32.20,2308736,276286,10673,2595696,88.94,10.64,0.41,51268,47586,1.08,1.6,4802,274182,1.75,2.2,5348119,73.27,15.28,2279.0,1700.5,18.2,2.14,0.6,91.59,8.41,5.48
1,2,2014,5.0,Minnesota,MN,MRO,East North Central,-0.1,normal,"Sunday, May 11, 2014",6:38:00 PM,"Sunday, May 11, 2014",6:39:00 PM,intentional attack,vandalism,NaN,1.0,NaN,NaN,12.12,9.71,6.49,9.28,1.59e+06,1.81e+06,1.89e+06,5.28e+06,30.03,34.21,35.73,2345860,284978,9898,2640737,88.83,10.79,0.37,53499,49091,1.09,1.9,5226,291955,1.79,2.2,5457125,73.27,15.28,2279.0,1700.5,18.2,2.14,0.6,91.59,8.41,5.48
2,3,2010,10.0,Minnesota,MN,MRO,East North Central,-1.5,cold,"Tuesday, October 26, 2010",8:00:00 PM,"Thursday, October 28, 2010",10:00:00 PM,severe weather,heavy wind,NaN,3000.0,NaN,70000.0,10.87,8.19,6.07,8.15,1.47e+06,1.80e+06,1.95e+06,5.22e+06,28.10,34.50,37.37,2300291,276463,10150,2586905,88.92,10.69,0.39,50447,47287,1.07,2.7,4571,267895,1.71,2.1,5310903,73.27,15.28,2279.0,1700.5,18.2,2.14,0.6,91.59,8.41,5.48
3,4,2012,6.0,Minnesota,MN,MRO,East North Central,-0.1,normal,"Tuesday, June 19, 2012",4:30:00 AM,"Wednesday, June 20, 2012",11:00:00 PM,severe weather,thunderstorm,NaN,2550.0,NaN,68200.0,11.79,9.25,6.71,9.19,1.85e+06,1.94e+06,1.99e+06,5.79e+06,31.99,33.54,34.44,2317336,278466,11010,2606813,88.90,10.68,0.42,51598,48156,1.07,0.6,5364,277627,1.93,2.2,5380443,73.27,15.28,2279.0,1700.5,18.2,2.14,0.6,91.59,8.41,5.48
4,5,2015,7.0,Minnesota,MN,MRO,East North Central,1.2,warm,"Saturday, July 18, 2015",2:00:00 AM,"Sunday, July 19, 2015",7:00:00 AM,severe weather,NaN,NaN,1740.0,250.0,250000.0,13.07,10.16,7.74,10.43,2.03e+06,2.16e+06,1.78e+06,5.97e+06,33.98,36.21,29.78,2374674,289044,9812,2673531,88.82,10.81,0.37,54431,49844,1.09,1.7,4873,292023,1.67,2.2,5489594,73.27,15.28,2279.0,1700.5,18.2,2.14,0.6,91.59,8.41,5.48


In [8]:
outage = original[['YEAR','MONTH','U.S._STATE','NERC.REGION', 'CLIMATE.REGION', 
                   'ANOMALY.LEVEL', 'CLIMATE.CATEGORY','OUTAGE.START.DATE', 'OUTAGE.START.TIME', 
                   'CAUSE.CATEGORY', 'CAUSE.CATEGORY.DETAIL','HURRICANE.NAMES', 'OUTAGE.DURATION', 
                   'DEMAND.LOSS.MW','CUSTOMERS.AFFECTED', 'RES.PRICE', 'COM.PRICE', 
                   'IND.PRICE','TOTAL.PRICE','TOTAL.CUSTOMERS', 'POPULATION', 'AREAPCT_URBAN', 'PCT_LAND']]

In [ ]:
outage.dropna(subset=['OUTAGE.DURATION'], inplace=True)

In [10]:
outage['CAUSE.CATEGORY.DETAIL']=outage['CAUSE.CATEGORY.DETAIL'].apply(lambda x: None if pd.isna(x) else x.strip())
outage['DEMAND.LOSS.MW']=outage['DEMAND.LOSS.MW'].apply(lambda x: np.nan if x==0 else x)
outage['CUSTOMERS.AFFECTED']=outage['CUSTOMERS.AFFECTED'].apply(lambda x: np.nan if x==0 else x)
outage['OUTAGE.START.DATE']=outage['OUTAGE.START.DATE'].apply(pd.to_datetime)
outage['OUTAGE.START.TIME']=outage['OUTAGE.START.TIME'].apply(lambda x: datetime.strptime(x,'%I:%M:%S %p').time())
outage['DAY.OR.NIGHT']=outage['OUTAGE.START.TIME'].apply(lambda x: x if pd.isna(x) else('DAY' if x>time(9) and x<time(17) else 'NIGHT'))
outage['HAS.HURRICANE']=outage['HURRICANE.NAMES'].notna()
outage=outage.drop('HURRICANE.NAMES', axis=1)

In [11]:
duration_level={(0, 60): 'Momentary', (61, 720): 'Short-term', (721, 1440): 'Extended', (1441, 4320): 'Prolonged', (4320, np.inf): 'Chronic'}

In [12]:
def duration_classify(duration):
    for range, level in duration_level.items():
        if range[0]<=duration<=range[1]:
            return level
    return 'Other'

In [13]:
outage['duration level']=outage['OUTAGE.DURATION'].apply(duration_classify)
outage.drop('OUTAGE.DURATION',axis=1, inplace=True)

In [14]:
from sklearn.preprocessing import PolynomialFeatures, StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score, f1_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer, SimpleImputer, KNNImputer
from sklearn.ensemble import RandomForestClassifier

In [15]:
#base line model
X_base=outage[['NERC.REGION', 'CLIMATE.REGION']]
y_base=outage['duration level']
X_train, X_test, y_train, y_test = train_test_split(X_base, y_base, test_size=0.2, random_state=42)
col_trans=ColumnTransformer(
    transformers=[
        ('one hot',OneHotEncoder(handle_unknown='ignore'), ['NERC.REGION', 'CLIMATE.REGION'])
    ],
    remainder='passthrough'
)
base_mdl=Pipeline(steps=[
    ('trains', col_trans),
    ('dt', DecisionTreeClassifier(max_depth=10))
])
base_mdl.fit(X_train, y_train)

Pipeline(steps=[('trains',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('one hot',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['NERC.REGION',
                                                   'CLIMATE.REGION'])])),
                ('dt', DecisionTreeClassifier(max_depth=10))])

In [16]:
#base line accuracy
y_pred = base_mdl.predict(X_test)
accuracy_score(y_test, y_pred)

0.3547297297297297

In [17]:
#base line f1
f1_score(y_test, y_pred, average='weighted')

np.float64(0.2938932035852283)

In [18]:
def creaete_fig(df, col1, col2, cat_num=1):
    if cat_num==1:
        fig_df=df[[col1, col2]].groupby(col1).mean().reset_index()
        return px.bar(fig_df, x=col1, y=col2)
    else:
        fig_df=df[[col1, col2]].groupby([col1, col2]).size().reset_index(name='size')
        return px.bar(fig_df, x=col1, y='size', color=col2, barmode='group')
creaete_fig(outage, 'duration level', 'POPULATION', cat_num=1).show()

In [19]:
#final model
X_final=outage[['NERC.REGION', 'CLIMATE.REGION', 'CAUSE.CATEGORY', 'CAUSE.CATEGORY.DETAIL', 'DEMAND.LOSS.MW', 'CUSTOMERS.AFFECTED', 'ANOMALY.LEVEL']]
y_final=outage['duration level']
X_train, X_test, y_train, y_test = train_test_split(X_final, y_final, test_size=0.2, random_state=42)
demand_trans=Pipeline(steps=[
    ('impute', IterativeImputer()),
    ('std', StandardScaler())
])
customer_trans=Pipeline(steps=[
    ('impute', KNNImputer(n_neighbors=5)),
    ('std', StandardScaler())
])
col_trans=ColumnTransformer(
    transformers=[
        ('demand impute', demand_trans, ['DEMAND.LOSS.MW']),
        ('customer impute', customer_trans, ['CUSTOMERS.AFFECTED']),
        ('one hot',OneHotEncoder(handle_unknown='ignore'), ['NERC.REGION','CLIMATE.REGION', 'CAUSE.CATEGORY', 'CAUSE.CATEGORY.DETAIL'])
    ],
    remainder='passthrough'
)
final_mdl=Pipeline(steps=[
    ('trains', col_trans),
    ('forest', RandomForestClassifier(max_depth=10))
])
final_mdl.fit(X_train, y_train)
y_pred = final_mdl.predict(X_test)
accuracy_score(y_test, y_pred)

0.527027027027027

In [20]:
#final with grid
grid={
    'forest__n_estimators':[50, 75, 100],
    'forest__max_depth':[5, 10, 50],
    'forest__min_samples_split':[3, 5, 10]
}
rf_g_search=GridSearchCV(estimator=final_mdl, param_grid=grid, cv=5, scoring='accuracy')
rf_g_search.fit(X_train, y_train)
y_pred=rf_g_search.best_estimator_.predict(X_test)
accuracy_score(y_test, y_pred)

0.5168918918918919

In [21]:
rf_g_search.best_params_

{'forest__max_depth': 10,
 'forest__min_samples_split': 5,
 'forest__n_estimators': 50}

In [113]:
test=original[['YEAR', 'U.S._STATE','NERC.REGION', 'CLIMATE.REGION', 'CAUSE.CATEGORY', 
               'CAUSE.CATEGORY.DETAIL', 'DEMAND.LOSS.MW', 'CUSTOMERS.AFFECTED', 'ANOMALY.LEVEL', 
               'PC.REALGSP.STATE', 'OUTAGE.DURATION', 'PC.REALGSP.REL', 'POPULATION']]
test['OUTAGE.DURATION']=test['OUTAGE.DURATION'].apply(lambda x: np.nan if x==0 else x)
test.dropna(subset=['OUTAGE.DURATION'], inplace=True)

In [126]:
test['serious']=((test['PC.REALGSP.STATE']*test['POPULATION']*test['OUTAGE.DURATION'])/3600).apply(np.log)

In [127]:
sorted=test.sort_values(by='serious').set_index(np.arange(test.shape[0]))

In [128]:
px.line(x=sorted.index, y=sorted['serious'])

In [145]:
serious_level={(0, 22): 'Minor', (22, 26): 'Normal', (26, 28): 'Serious', (28, np.inf): 'Critical'}
def serious_classify(num):
    for range, level in serious_level.items():
        if range[0]<=num<=range[1]:
            return level
    return 'Other'

In [146]:
test['serious level']=test['serious'].apply(serious_classify)

In [147]:
test['serious']

0       26.17
1       18.21
2       26.13
        ...  
1529    22.78
1531    20.21
1532    21.33
Name: serious, Length: 1398, dtype: float64

In [148]:
px.bar(test, x='serious level')

In [149]:
test_x=test[['YEAR', 'U.S._STATE','NERC.REGION', 'CLIMATE.REGION', 'CAUSE.CATEGORY', 'CAUSE.CATEGORY.DETAIL', 'DEMAND.LOSS.MW', 'CUSTOMERS.AFFECTED', 'ANOMALY.LEVEL', 'PC.REALGSP.STATE']]
test_y=test['serious level']
X_train, X_test, y_train, y_test = train_test_split(test_x, test_y, test_size=0.2, random_state=42)
demand_trans=Pipeline(steps=[
    ('impute', IterativeImputer()),
    ('std', StandardScaler())
])
customer_trans=Pipeline(steps=[
    ('impute', KNNImputer(n_neighbors=5)),
    ('std', StandardScaler())
])
col_trans=ColumnTransformer(
    transformers=[
        ('demand impute', demand_trans, ['DEMAND.LOSS.MW']),
        ('customer impute', customer_trans, ['CUSTOMERS.AFFECTED']),
        ('one hot',OneHotEncoder(handle_unknown='ignore'), ['U.S._STATE', 'NERC.REGION','CLIMATE.REGION', 'CAUSE.CATEGORY', 'CAUSE.CATEGORY.DETAIL'])
    ],
    remainder='passthrough'
)
final_mdl=Pipeline(steps=[
    ('trains', col_trans),
    ('forest', RandomForestClassifier(max_depth=10))
])
final_mdl.fit(X_train, y_train)
y_pred = final_mdl.predict(X_test)
accuracy_score(y_test, y_pred)

0.6607142857142857

In [90]:
outage

,YEAR,MONTH,U.S._STATE,NERC.REGION,CLIMATE.REGION,ANOMALY.LEVEL,CLIMATE.CATEGORY,OUTAGE.START.DATE,OUTAGE.START.TIME,CAUSE.CATEGORY,CAUSE.CATEGORY.DETAIL,DEMAND.LOSS.MW,CUSTOMERS.AFFECTED,RES.PRICE,COM.PRICE,IND.PRICE,TOTAL.PRICE,TOTAL.CUSTOMERS,POPULATION,AREAPCT_URBAN,PCT_LAND,DAY.OR.NIGHT,HAS.HURRICANE,duration level
0,2011,7.0,Minnesota,MRO,East North Central,-0.3,normal,2011-07-01,17:00:00,severe weather,None,NaN,70000.0,11.60,9.18,6.81,9.28,2595696,5348119,2.14,91.59,NIGHT,False,Prolonged
1,2014,5.0,Minnesota,MRO,East North Central,-0.1,normal,2014-05-11,18:38:00,intentional attack,vandalism,NaN,NaN,12.12,9.71,6.49,9.28,2640737,5457125,2.14,91.59,NIGHT,False,Momentary
2,2010,10.0,Minnesota,MRO,East North Central,-1.5,cold,2010-10-26,20:00:00,severe weather,heavy wind,NaN,70000.0,10.87,8.19,6.07,8.15,2586905,5310903,2.14,91.59,NIGHT,False,Prolonged
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1529,2011,12.0,North Dakota,MRO,West North Central,-0.9,cold,2011-12-06,08:00:00,public appeal,None,155.0,34500.0,8.41,7.80,6.20,7.56,394394,685326,0.27,97.60,NIGHT,False,Short-term
1531,2009,8.0,South Dakota,RFC,West North Central,0.5,warm,2009-08-29,22:54:00,islanding,None,84.0,NaN,9.25,7.47,5.53,7.67,436229,807067,0.30,98.31,NIGHT,False,Momentary
1532,2009,8.0,South Dakota,MRO,West North Central,0.5,warm,2009-08-29,11:00:00,islanding,None,373.0,NaN,9.25,7.47,5.53,7.67,436229,807067,0.30,98.31,DAY,False,Short-term


In [99]:
grid={
    'forest__n_estimators':[25, 50],
    'forest__max_depth':[10, 30, 50],
    'forest__min_samples_split':[20, 30]
}
rf_g_search=GridSearchCV(estimator=final_mdl, param_grid=grid, cv=5, scoring='accuracy')
rf_g_search.fit(X_train, y_train)
y_pred=rf_g_search.best_estimator_.predict(X_test)
accuracy_score(y_test, y_pred)

/Users/the87/miniforge3/envs/dsc80/lib/python3.12/site-packages/sklearn/model_selection/_split.py:776: UserWarning:

The least populated class in y has only 3 members, which is less than n_splits=5.



0.6321428571428571

In [105]:
test_x=test[['YEAR', 'U.S._STATE']]
test_y=test['serious level']
X_train, X_test, y_train, y_test = train_test_split(test_x, test_y, test_size=0.2, random_state=42)
demand_trans=Pipeline(steps=[
    ('impute', IterativeImputer()),
    ('std', StandardScaler())
])
customer_trans=Pipeline(steps=[
    ('impute', KNNImputer(n_neighbors=5)),
    ('std', StandardScaler())
])
col_trans=ColumnTransformer(
    transformers=[
        ('one hot',OneHotEncoder(handle_unknown='ignore'), ['YEAR', 'U.S._STATE'])
    ],
    remainder='passthrough'
)
final_mdl=Pipeline(steps=[
    ('trains', col_trans),
    ('forest', RandomForestClassifier(max_depth=10))
])
final_mdl.fit(X_train, y_train)
y_pred = final_mdl.predict(X_test)
accuracy_score(y_test, y_pred)

0.48928571428571427